## Feature Selection

### EDA-driven reasoning for base features:

1. Price Features (Close, High, Low, Open) have very low correlation with y_return and are non-stationary — dropped as raw features.
2. Returnlag, log_return and OC_change are redundant (cross-correlation ~0.998 / 0.814). Kept as Ret_Lag1 for naming consistency.
3. HL_range and Volatility_5 overlap moderately (0.678) but each captures a different dimension (intraday vs rolling). Both defensible to keep.
4. Volume and Volume_log have near-zero correlation with y_return (~0.003). Dropped.
5. Price level can be made stationary and meaningful via ratio: (Close - MA_5) / MA_5 = Momentum_Deviation.
6. Ticker identity is relevant — label encoded with sorted mapping for reproducibility.

### Additional features from teammate's engineering:
RSI, MACD_Norm, ATR_Pct, Ret_Lag2, BB_Pct, Price_vs_SMA50 are industry-standard stationary technical indicators that add signal not captured by the base set.
Rel_Volume and OC_Change are dropped — Rel_Volume shows weak signal, OC_Change is redundant with Ret_Lag1.

### Final feature set:
1. **Ret_Lag1** — 1-day lagged return (momentum)
2. **Ret_Lag2** — 2-day lagged return (extended momentum)
3. **RSI** — Relative Strength Index, momentum oscillator (0-100)
4. **MACD_Norm** — MACD normalized by price, trend momentum signal
5. **ATR_Pct** — Average True Range as % of price, normalized volatility
6. **HL_range** — Intraday High-Low range normalized by Close
7. **BB_Pct** — Bollinger Band %B, mean reversion signal
8. **Price_vs_SMA50** — Price relative to 50-day MA, trend strength
9. **Momentum_Deviation** — (Close - MA_5) / MA_5, short-term trend deviation
10. **Encoded_Ticker** — Label-encoded ticker identity

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/processed/sp500_panel.csv")
df["Date"] = pd.to_datetime(df["Date"])

# Sort by Ticker + Date — critical for all groupby time-series operations
df = df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

print(f"Loaded: {len(df)} rows across {df['Ticker'].nunique()} tickers")
df.head()

Loaded: 1218220 rows across 501 tickers


,Date,Close,High,Low,Open,Volume,Ticker,y_return,y_direction,Returnlag,log_return,HL_range,OC_change,Volume_log,MA_5,Volatility_5
0,2015-12-28,38.539181,38.815913,38.299350,38.760566,1458200,A,0.013882,1,-0.008543,-0.008580,0.013404,-0.005712,14.192714,38.325180,0.009196
1,2015-12-29,39.074196,39.184887,38.668324,38.815916,1757000,A,-0.004485,0,0.013882,0.013787,0.013220,0.006654,14.379119,38.607442,0.010441
2,2015-12-30,38.898933,39.092647,38.751345,39.074198,834300,A,-0.005826,0,-0.004485,-0.004495,0.008774,-0.004485,13.634350,38.782706,0.009940
3,2015-12-31,38.672314,39.171786,38.589068,38.755560,1451000,A,-0.026788,0,-0.005826,-0.005843,0.015068,-0.002148,14.187764,38.811177,0.014453
4,2016-01-04,37.636372,38.098849,37.312639,37.978607,3287300,A,-0.003441,0,-0.026788,-0.027153,0.020890,-0.009011,15.005577,38.564199,0.014440


## Existing features from preprocessing
HL_range, Volatility_5, MA_5 already exist in the processed CSV.
We compute Momentum_Deviation from MA_5 here.

In [2]:
# Momentum Deviation — price vs short-term trend (5-day)
df["Momentum_Deviation"] = (df["Close"] - df["MA_5"]) / df["MA_5"]

print("Momentum_Deviation added.")
df[["Date", "Ticker", "Momentum_Deviation"]].head()

Momentum_Deviation added.


,Date,Ticker,Momentum_Deviation
0,2015-12-28,A,0.005584
1,2015-12-29,A,0.012090
2,2015-12-30,A,0.002997
3,2015-12-31,A,-0.003578
4,2016-01-04,A,-0.024059


## Additional feature engineering
Industry-standard stationary technical indicators added to enrich the feature pool.

In [3]:
# --- FEATURE GROUP 1: MOMENTUM ---

# RSI (Relative Strength Index) — momentum oscillator, bounded 0-100, stationary
def calculate_rsi(series, period=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

df["RSI"] = df.groupby("Ticker")["Close"].transform(calculate_rsi)

# MACD Normalized — trend momentum, normalized by price so comparable across all 501 tickers
def calculate_macd_norm(series):
    exp1 = series.ewm(span=12, adjust=False).mean()
    exp2 = series.ewm(span=26, adjust=False).mean()
    macd = exp1 - exp2
    return macd / series

df["MACD_Norm"] = df.groupby("Ticker")["Close"].transform(calculate_macd_norm)

# Lagged Returns — Ret_Lag1 is same computation as Returnlag in preprocessing,
# using consistent naming. Ret_Lag2 adds a second lag dimension.
df["Ret_Lag1"] = df.groupby("Ticker")["Close"].pct_change()
df["Ret_Lag2"] = df.groupby("Ticker")["Ret_Lag1"].shift(1)


# --- FEATURE GROUP 2: VOLATILITY ---

# ATR Normalized — true range accounts for overnight gaps, more complete than HL_range alone
def calculate_atr_pct(group):
    high = group["High"]
    low = group["Low"]
    close = group["Close"]
    prev_close = close.shift(1)
    tr1 = high - low
    tr2 = (high - prev_close).abs()
    tr3 = (low - prev_close).abs()
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    atr = tr.rolling(window=14).mean()
    return atr / close

df["ATR_Pct"] = (
    df.groupby("Ticker")
    .apply(calculate_atr_pct, include_groups=False)
    .reset_index(level=0, drop=True)
)


# --- FEATURE GROUP 3: TREND & MEAN REVERSION ---

# Price vs 50-day SMA — longer-term trend strength, complements Momentum_Deviation (5-day)
df["Price_vs_SMA50"] = df["Close"] / df.groupby("Ticker")["Close"].transform(
    lambda x: x.rolling(50).mean()
)

# Bollinger Band %B — where is price within the bands? mean reversion signal
def calculate_bb_pct(series, length=20, std=2):
    mean = series.rolling(length).mean()
    std_dev = series.rolling(length).std()
    upper = mean + (std_dev * std)
    lower = mean - (std_dev * std)
    return (series - lower) / (upper - lower)

df["BB_Pct"] = df.groupby("Ticker")["Close"].transform(calculate_bb_pct)


# --- CLEANUP ---
original_len = len(df)
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Feature Engineering Complete. Dropped {original_len - len(df)} rows (NaNs from rolling windows).")
print(f"Final dataset shape: {df.shape}")

Feature Engineering Complete. Dropped 24720 rows (NaNs from rolling windows).
Final dataset shape: (1193500, 24)


## Ticker Label Encoding
Sorted encoding ensures reproducibility — same ticker always maps to same integer regardless of load order.

In [4]:
# Sorted for reproducibility — unsorted .unique() order can vary by run
ticker_list = sorted(df["Ticker"].unique())
ticker_dict = {ticker: i for i, ticker in enumerate(ticker_list)}
df["Encoded_Ticker"] = df["Ticker"].map(ticker_dict)

print(f"Encoded {len(ticker_list)} tickers.")
df[["Ticker", "Encoded_Ticker"]].drop_duplicates().head(10)

Encoded 499 tickers.


,Ticker,Encoded_Ticker
0,A,0
2461,AAPL,1
4922,ABBV,2
7383,ABNB,3
8592,ABT,4
11053,ACGL,5
13514,ACN,6
15975,ADBE,7
18436,ADI,8
20897,ADM,9


## Final Feature Selection & Save
Selected features cover four distinct dimensions:
- **Momentum**: Ret_Lag1, Ret_Lag2, RSI, MACD_Norm
- **Volatility**: ATR_Pct, HL_range
- **Trend**: Price_vs_SMA50, Momentum_Deviation, BB_Pct
- **Identity**: Encoded_Ticker

Dropped from candidate pool:
- Rel_Volume — near-zero signal (like raw Volume_log at 0.003 correlation)
- OC_Change — redundant with Ret_Lag1 (correlation ~0.814)
- Volatility_5 — ATR_Pct is a strictly better normalized volatility measure
- Returnlag / log_return — superseded by Ret_Lag1 + Ret_Lag2 with consistent naming

In [5]:
FINAL_FEATURES = [
    "Date",
    # Momentum
    "Ret_Lag1",
    "Ret_Lag2",
    "RSI",
    "MACD_Norm",
    # Volatility
    "ATR_Pct",
    "HL_range",
    # Trend & Mean Reversion
    "Price_vs_SMA50",
    "Momentum_Deviation",
    "BB_Pct",
    # Identity
    "Encoded_Ticker",
    # Target
    "y_return",
]

ndf = df[FINAL_FEATURES].copy()

print(f"Final shape: {ndf.shape}")
print(f"Features: {[f for f in FINAL_FEATURES if f not in ['Date', 'y_return']]}")
ndf.head()

Final shape: (1193500, 12)
Features: ['Ret_Lag1', 'Ret_Lag2', 'RSI', 'MACD_Norm', 'ATR_Pct', 'HL_range', 'Price_vs_SMA50', 'Momentum_Deviation', 'BB_Pct', 'Encoded_Ticker']


,Date,Ret_Lag1,Ret_Lag2,RSI,MACD_Norm,ATR_Pct,HL_range,Price_vs_SMA50,Momentum_Deviation,BB_Pct,Encoded_Ticker,y_return
0,2016-03-09,0.001050,-0.033984,57.966095,0.007981,0.020606,0.011540,1.003275,-0.018230,0.596292,0,-0.003934
1,2016-03-10,-0.003934,0.001050,54.655204,0.006779,0.020951,0.023433,1.001271,-0.015909,0.541852,0,0.032912
2,2016-03-11,0.032912,-0.003934,59.287856,0.008094,0.021267,0.024216,1.035871,0.017059,0.845679,0,-0.005353
3,2016-03-14,-0.005353,0.032912,65.920810,0.008822,0.020905,0.019477,1.031980,0.013770,0.771143,0,-0.018708
4,2016-03-15,-0.018708,-0.005353,56.500849,0.007913,0.021136,0.016453,1.014563,-0.006229,0.527829,0,0.018021


In [6]:
import os
os.makedirs("data/features", exist_ok=True)
ndf.to_csv("data/features/sp500_panel_with_features.csv", index=False)

if os.path.exists("data/features/sp500_panel_with_features.csv"):
    print("SUCCESS: Feature set saved to data/features/sp500_panel_with_features.csv")
    print("Ready for model training with 80-20 time-series split on Date column.")

SUCCESS: Feature set saved to data/features/sp500_panel_with_features.csv
Ready for model training with 80-20 time-series split on Date column.
